<a href="https://colab.research.google.com/github/Nameisharisai/VIT-SysRadNet/blob/main/VIT_SysRadnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# First, let's make sure we have all the necessary tools installed!
# We're installing libraries like 'timm' for vision transformers, 'torchmetrics' for evaluation,
# 'scikit-learn' for machine learning utilities, 'matplotlib' and 'seaborn' for plotting,
# 'pandas' and 'numpy' for data handling, and 'Pillow' for image processing.
!pip install -q timm torchmetrics scikit-learn matplotlib seaborn pandas numpy Pillow tqdm
# We also need 'opencv-python-headless' for image manipulation without a GUI, and 'kaggle'
# to download our dataset easily.
!pip install -q opencv-python-headless kaggle

# These are standard Python modules for file operations, JSON handling, and managing warnings.
import os, json, shutil, warnings
warnings.filterwarnings('ignore') # Let's hide annoying warnings for a cleaner output.

print(" Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 297.4 kB/s eta 0:00:00
 Dependencies installed


In [ ]:
from google.colab import files # This allows us to easily upload files in Colab.
import os, shutil # Standard modules for operating system interactions and file operations.

print("Step 1: Upload your kaggle.json file")
# Kaggle requires authentication to download datasets. The 'kaggle.json' file contains your credentials.
# This line prompts you to upload the file from your local machine.
uploaded = files.upload()

# We create a directory where Kaggle expects to find its configuration file.
os.makedirs('/root/.kaggle', exist_ok=True)
# Then, we copy the uploaded 'kaggle.json' into that directory.
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
# This command sets the correct permissions for the Kaggle API key, making it secure (read-only for the owner).
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print(" kaggle.json configured")

print("\n Step 2: Upload cbis_ddsm_comprehensive_master.csv")
# Now, we'll upload the CSV file that contains metadata and radiomic features for our dataset.
uploaded_csv = files.upload()
# We extract the name of the uploaded CSV file, assuming only one was uploaded.
CSV_PATH = list(uploaded_csv.keys())[0]
print(f" CSV loaded: {CSV_PATH}")

📂 Step 1: Upload your kaggle.json file


Saving kaggle.json to kaggle.json
✅ kaggle.json configured

📂 Step 2: Upload cbis_ddsm_comprehensive_master.csv


Saving cbis_ddsm_comprehensive_master.csv to cbis_ddsm_comprehensive_master.csv
✅ CSV loaded: cbis_ddsm_comprehensive_master.csv


In [ ]:
import glob # Used for finding files matching a pattern.
import os # For interacting with the operating system, like creating directories.

print("📥 Downloading CBIS-DDSM dataset from Kaggle...")
# Ensure a clean directory for our dataset. We remove any previous data to avoid conflicts.
!rm -rf /content/cbis_data
# Create a new, empty directory to store the downloaded dataset.
os.makedirs('/content/cbis_data', exist_ok=True)

# Using the standard CBIS-DDSM dataset slug, we download and unzip the dataset directly into '/content/cbis_data'.
# This can take a few minutes as it's a large dataset.
!kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset -p /content/cbis_data --unzip

# After downloading, we search for all image files (PNG and JPG) within the newly created directory.
# 'recursive=True' makes sure it searches through all subfolders.
image_paths = glob.glob('/content/cbis_data/**/*.png', recursive=True) + \
              glob.glob('/content/cbis_data/**/*.jpg', recursive=True)

print(f"✅ Found {len(image_paths)} images")
# A quick check to see if images were found. If not, it suggests troubleshooting steps.
if len(image_paths) == 0:
    print("❌ Still no images found. Please check if the Kaggle dataset slug is correct and you have accepted the competition rules if applicable.")

📥 Downloading CBIS-DDSM dataset from Kaggle...
Dataset URL: https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset
License(s): CC-BY-SA-3.0
100% 4.95G/4.95G [00:28<00:00, 188MB/s]

✅ Found 10237 images


In [ ]:
import torch # The core PyTorch library for deep learning.
import torch.nn as nn # Neural network modules and layers.
import torch.nn.functional as F # Functional interface for common operations.
from torch.utils.data import Dataset, DataLoader # Utilities for handling datasets and loading data in batches.
from torch.optim import AdamW # The AdamW optimizer for training neural networks.
from torch.optim.lr_scheduler import CosineAnnealingLR # Learning rate scheduler.
import timm # PyTorch Image Models, providing pre-trained models like Vision Transformers.
import pandas as pd # For data manipulation and analysis.
import numpy as np # For numerical operations, especially array manipulation.
import cv2 # OpenCV library for image processing (imported as 'opencv-python-headless').
import matplotlib # For creating static, interactive, and animated visualizations.
matplotlib.use('Agg') # Use the 'Agg' backend to allow Matplotlib to run without a display server.
import matplotlib.pyplot as plt # For plotting data.
import seaborn as sns # For making attractive and informative statistical graphics.
from PIL import Image # Python Imaging Library for image processing.
from pathlib import Path # Object-oriented filesystem paths.
from sklearn.model_selection import StratifiedKFold # For creating stratified cross-validation folds.
from sklearn.metrics import (roc_auc_score, accuracy_score, # Metrics for model evaluation.
    precision_recall_fscore_support, roc_curve,
    precision_recall_curve, confusion_matrix)
from sklearn.calibration import calibration_curve # For assessing model calibration.
from tqdm import tqdm # For displaying smart progress bars.
import random, glob # For generating random numbers and finding files.

# Setting a random seed for reproducibility. This ensures that our experiments can be replicated.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# If a CUDA-enabled GPU is available, we also set the seed for GPU operations.
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# The Configuration class (CFG) holds all the hyperparameters and paths for our model and training.
# It's a convenient way to manage all settings in one place.
class CFG:
    IMG_SIZE     = 512    # The desired size for our mammogram images (width and height).
    PATCH_SIZE   = 16     # Patch size for the Vision Transformer model.
    EMBED_DIM    = 768    # Dimension of the embedding space for our features.
    DEPTH        = 12     # Number of transformer blocks.
    NUM_HEADS    = 12     # Number of attention heads in the transformer.
    DROPOUT      = 0.1    # Dropout rate to prevent overfitting.
    RADIOMIC_DIM = 506    # Placeholder for the dimension of radiomic features; will be updated.
    MARGIN       = 1.0    # Margin for the symmetry loss function.
    BATCH_SIZE   = 4      # Number of samples processed in each training iteration.
    EPOCHS       = 30     # Total number of training cycles over the dataset.
    LR           = 1e-4   # Learning rate for the optimizer.
    WEIGHT_DECAY = 1e-4   # Weight decay (L2 regularization) for the optimizer.
    N_FOLDS      = 5      # Number of folds for k-fold cross-validation.
    LAMBDA_CE    = 0.50   # Weight for the Cross-Entropy loss component.
    LAMBDA_SYM   = 0.25   # Weight for the Symmetry loss component.
    LAMBDA_CAL   = 0.15   # Weight for the Calibration loss component.
    LAMBDA_AUX   = 0.10   # Weight for any auxiliary loss component.
    MC_SAMPLES   = 10     # Number of Monte Carlo samples (if used for uncertainty).
    CSV_PATH     = CSV_PATH # Path to the CSV file containing patient data and features.
    IMG_ROOT     = '/content/cbis_data' # Root directory where mammogram images are stored.
    FIGURE_DIR   = '/content/figures'   # Directory to save generated figures and plots.
    CKPT_DIR     = '/content/checkpoints' # Directory to save model checkpoints.

# Create directories for figures and model checkpoints if they don't already exist.
os.makedirs(CFG.FIGURE_DIR, exist_ok=True)
os.makedirs(CFG.CKPT_DIR, exist_ok=True)

# Determine the device to run the model on (GPU if available, otherwise CPU).
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Config ready | Device: {DEVICE}")
# Print GPU details if a GPU is being used.
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

✅ Config ready | Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB


In [ ]:
# This function loads patient data from a CSV and organizes it into 'exam' objects.
# Each exam object will represent a patient's study, containing their label, radiomic features,
# and paths to their mammogram images for different views (e.g., LCC, LMLO, RCC, RMLO).
def load_and_build_patient_exams(csv_path, img_root):
    # Read the CSV file into a pandas DataFrame.
    df = pd.read_csv(csv_path)
    print(f"CSV shape: {df.shape}")

    # Identify columns that contain radiomic features (those starting with 'feat_').
    feat_cols = [c for c in df.columns if c.startswith('feat_')]
    print(f"Radiomic feature columns: {len(feat_cols)}")

    # Try to find the column that indicates the diagnosis/label (e.g., 'pathology', 'label').
    label_col = None
    for candidate in ['pathology', 'label', 'assessment', 'class']:
        if candidate in df.columns:
            label_col = candidate; break
    # If common names aren't found, assume the last non-feature column is the label.
    if label_col is None:
        label_col = [c for c in df.columns if not c.startswith('feat_')][-1]
    print(f"Label column: '{label_col}'")

    # Convert the diagnosis into a binary label: 1 for malignant, 0 for benign.
    df['binary_label'] = df[label_col].apply(
        lambda x: 1 if str(x).upper() in ['MALIGNANT','1','POSITIVE','CANCER','M'] else 0)

    # Identify columns for patient ID, breast side (left/right), and image view.
    side_col = next((c for c in ['left or right breast','side','breast_side','laterality'] if c in df.columns), None)
    view_col = next((c for c in ['image view','view','image_view','projection'] if c in df.columns), None)
    pid_col  = next((c for c in ['patient_id','patient id','case_id','id'] if c in df.columns), None)
    print(f"Patient col: {pid_col}, Side col: {side_col}, View col: {view_col}")

    # Get a list of all image paths on disk.
    all_images = (glob.glob(f'{img_root}/**/*.png', recursive=True) +
                  glob.glob(f'{img_root}/**/*.jpg', recursive=True))
    print(f"Total images on disk: {len(all_images)}")

    # Create a lookup dictionary for image paths, using the filename stem (without extension) as key.
    img_lookup = {Path(p).stem.lower(): p for p in all_images}
    exams = [] # This list will store our structured exam objects.

    # If patient ID, side, and view columns are available, we can group data by patient.
    if pid_col and side_col and view_col:
        # Iterate through each patient's data.
        for pid, group in df.groupby(pid_col):
            # Separate data for left and right breasts.
            left_rows  = group[group[side_col].str.upper().str.contains('LEFT',  na=False)]
            right_rows = group[group[side_col].str.upper().str.contains('RIGHT', na=False)]

            # Calculate mean radiomic features for left and right breasts. If no features or rows, use zeros.
            r_L = left_rows[feat_cols].mean().values.astype(np.float32)  if (len(feat_cols)>0 and len(left_rows)>0)  else np.zeros(max(len(feat_cols),1), np.float32)
            r_R = right_rows[feat_cols].mean().values.astype(np.float32) if (len(feat_cols)>0 and len(right_rows)>0) else np.zeros(max(len(feat_cols),1), np.float32)

            # Get the overall label for the patient (malignant if any finding is malignant).
            label = int(group['binary_label'].max())

            # Initialize image paths for the four standard mammogram views to None.
            view_map = {'LCC': None, 'LMLO': None, 'RCC': None, 'RMLO': None}
            # Iterate through each row (image/finding) for the current patient.
            for _, row in group.iterrows():
                side = str(row.get(side_col, '')).upper()
                view = str(row.get(view_col, '')).upper().replace(' ','').replace('-','').replace('_','')
                key  = ('L' if 'LEFT' in side else 'R') + view[:3] # Construct a key like 'LCC', 'RMLO'
                key  = key[:4]
                img_path = None
                # Try to find the corresponding image path on disk based on patient ID and view.
                for stem, path in img_lookup.items():
                    pid_str = str(pid).lower().replace(' ','_').replace('-','_')
                    if pid_str in stem or str(pid).lower() in stem:
                        if view[:2] in stem.upper() or side[:1] in stem.upper():
                            img_path = path; break
                # Assign the found image path to the correct view in the map.
                if key in view_map:
                    view_map[key] = img_path

            # Add the complete exam object for the patient.
            exams.append({'patient_id': pid, 'label': label,
                          'r_L': r_L, 'r_R': r_R,
                          'img_LCC': view_map['LCC'], 'img_LMLO': view_map['LMLO'],
                          'img_RCC': view_map['RCC'], 'img_RMLO': view_map['RMLO']})
    else:
        # If patient-level grouping is not possible, fall back to row-level processing (less ideal).
        print("☀ဤ Using row-level fallback")
        for idx, row in df.iterrows():
            # Use radiomic features from the current row, or zeros if none exist.
            r = row[feat_cols].values.astype(np.float32) if feat_cols else np.zeros(CFG.RADIOMIC_DIM, np.float32)
            # Assign a random image path if available, cycling through all_images.
            img_path = all_images[idx % len(all_images)] if all_images else None
            exams.append({'patient_id': idx, 'label': int(row['binary_label']),
                          'r_L': r, 'r_R': r,
                          'img_LCC': img_path, 'img_LMLO': None,
                          'img_RCC': None,      'img_RMLO': None})

    print(f"\n✅ Built {len(exams)} exam objects")
    # Count and print the distribution of malignant and benign cases.
    lbls = [e['label'] for e in exams]
    print(f"   Malignant: {sum(lbls)} | Benign: {len(lbls)-sum(lbls)}")
    return exams, len(feat_cols)

# Load the exams and update the RADIOMIC_DIM in our configuration.
exams, radiomic_dim = load_and_build_patient_exams(CFG.CSV_PATH, CFG.IMG_ROOT)
CFG.RADIOMIC_DIM = max(radiomic_dim, 1)

# Normalise radiomic features for better model training.
# Combine all left and right radiomic features.
all_r    = np.vstack([np.concatenate([e['r_L'], e['r_R']]) for e in exams])
# Calculate mean and standard deviation for normalization.
rad_mean = all_r.mean(0); rad_std = all_r.std(0) + 1e-8 # Add a small epsilon to avoid division by zero.
# Apply Z-score normalization to each exam's radiomic features.
for e in exams:
    full   = np.concatenate([e['r_L'], e['r_R']])
    normed = (full - rad_mean) / rad_std
    e['r_L'] = normed[:CFG.RADIOMIC_DIM].astype(np.float32)
    e['r_R'] = normed[CFG.RADIOMIC_DIM:].astype(np.float32)
print("✅ Radiomic features normalised")

CSV shape: (3568, 776)
Radiomic feature columns: 768
Label column: 'pathology'
Patient col: patient_id, Side col: left or right breast, View col: image view
Total images on disk: 10237

✅ Built 1566 exam objects
   Malignant: 752 | Benign: 814
✅ Radiomic features normalised


In [ ]:
import torchvision.transforms.functional as TF # Functions for image transformations, often used with PyTorch.

# This function preprocesses a single mammogram image.
# It handles loading, cleaning, enhancing, resizing, and normalizing the image.
def preprocess_mammogram(img_path, img_size=512, augment=False):
    # If the image path is invalid or doesn't exist, return a tensor of zeros.
    if img_path is None or not os.path.exists(str(img_path)):
        return torch.zeros(3, img_size, img_size)
    # Read the image in grayscale using OpenCV.
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    if img is None: # If image loading failed, return zeros.
        return torch.zeros(3, img_size, img_size)

    # Apply Otsu's thresholding to create a binary mask, helping to isolate the breast region.
    _, mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Apply morphological closing to the mask to fill small holes and smooth contours.
    kernel  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15,15))
    mask    = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    # Apply the mask to the original image, effectively removing background noise.
    img     = cv2.bitwise_and(img, img, mask=mask)

    # Denoise the image using a Gaussian blur to reduce high-frequency noise.
    img = cv2.GaussianBlur(img, (3,3), 0.5)
    # Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) to enhance local contrast.
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img   = clahe.apply(img)

    # Resize the image to the target size using linear interpolation.
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_LINEAR)
    # Normalize pixel values to a 0-1 range.
    img = img.astype(np.float32) / 255.0
    # Apply Z-score normalization (mean 0, std 1) and clip values to [-3, 3] to handle outliers.
    img = (img - img.mean()) / (img.std() + 1e-8)
    img = np.clip(img, -3, 3)
    # Rescale values to 0-1 after clipping.
    img = (img + 3) / 6.0
    # Convert the single-channel grayscale image into a 3-channel (RGB-like) tensor.
    tensor = torch.from_numpy(np.stack([img,img,img], axis=0)).float()

    # Apply data augmentation if 'augment' is True (used during training to increase data variability).
    if augment:
        if random.random() > 0.5: tensor = TF.hflip(tensor) # Random horizontal flip.
        tensor = TF.rotate(tensor, random.uniform(-15,15)) # Random rotation.
        tensor = TF.adjust_brightness(tensor, 1+random.uniform(-0.1,0.1)) # Random brightness adjustment.
        tensor = TF.adjust_contrast(tensor,   1+random.uniform(-0.1,0.1)) # Random contrast adjustment.
        tensor = (tensor + torch.randn_like(tensor)*0.01).clamp(0,1) # Add slight random noise.
    return tensor

# The CBISDDSMDataset class inherits from torch.utils.data.Dataset,
# which is essential for working with PyTorch's DataLoader.
class CBISDDSMDataset(Dataset):
    # The constructor takes the list of 'exams' (patient data) and an 'augment' flag.
    def __init__(self, exams, augment=False):
        self.exams = exams; self.augment = augment
    # Returns the total number of samples (exams) in the dataset.
    def __len__(self): return len(self.exams)
    # This method is called by DataLoader to fetch a single sample by its index.
    def __getitem__(self, idx):
        e = self.exams[idx] # Get the exam object for the current index.
        # Preprocess each of the four mammogram views and stack them into a single tensor.
        views = torch.stack([
            preprocess_mammogram(e['img_LCC'],  CFG.IMG_SIZE, self.augment),
            preprocess_mammogram(e['img_LMLO'], CFG.IMG_SIZE, self.augment),
            preprocess_mammogram(e['img_RCC'],  CFG.IMG_SIZE, self.augment),
            preprocess_mammogram(e['img_RMLO'], CFG.IMG_SIZE, self.augment),
        ])
        # Return the processed views, left radiomic features, right radiomic features, and the binary label.
        return (views,
                torch.tensor(e['r_L'], dtype=torch.float32),
                torch.tensor(e['r_R'], dtype=torch.float32),
                torch.tensor(e['label'], dtype=torch.float32))

print("✅ Dataset class ready")

✅ Dataset class ready


In [ ]:
class RadiomicsFusionBranch(nn.Module):
    def __init__(self, radiomic_dim, embed_dim, dropout=0.1):
        super().__init__()
        # This sequential network processes the concatenated left and right radiomic features.
        # It uses Linear layers for transformations, GELU for non-linearity, and Dropout for regularization.
        self.net = nn.Sequential(
            nn.Linear(radiomic_dim*2, embed_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim*2),    nn.GELU())
        # Separate projection layers for left and right radiomic features after initial processing.
        self.left_proj  = nn.Linear(embed_dim*2, embed_dim)
        self.right_proj = nn.Linear(embed_dim*2, embed_dim)

    # In the forward pass, left and right radiomic features are concatenated, passed through the 'net',
    # and then projected separately to get 'tok_L' and 'tok_R' tokens.
    def forward(self, r_L, r_R):
        h = self.net(torch.cat([r_L, r_R], dim=-1))
        return self.left_proj(h), self.right_proj(h)

class SymmetryModule(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        # Learnable symmetry tokens for CC and MLO views. These will interact with visual embeddings.
        self.sym_CC  = nn.Parameter(torch.randn(1,1,embed_dim)*0.02)
        self.sym_MLO = nn.Parameter(torch.randn(1,1,embed_dim)*0.02)
        # Multihead attention layers to model relationships between paired views (e.g., LCC and RCC).
        self.attn_CC  = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.attn_MLO = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        # Layer normalization layers for stable training.
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    # This module takes visual embeddings from paired views (LCC/RCC, LMLO/RMLO) and computes symmetry tokens.
    # These tokens capture shared information or differences between the corresponding left and right views.
    def forward(self, z_LCC, z_LMLO, z_RCC, z_RMLO):
        B = z_LCC.size(0) # Batch size.
        # Concatenate LCC and RCC embeddings to form key-value pairs for CC attention.
        kv_CC  = torch.stack([z_LCC, z_RCC],   dim=1)
        # The learnable sym_CC token queries the concatenated visual embeddings.
        e_CC,_ = self.attn_CC(self.sym_CC.expand(B,-1,-1), kv_CC, kv_CC)
        # Apply layer normalization and squeeze the result.
        e_CC   = self.norm1(e_CC.squeeze(1))

        # Similar process for MLO views (LMLO and RMLO).
        kv_MLO  = torch.stack([z_LMLO, z_RMLO], dim=1)
        e_MLO,_ = self.attn_MLO(self.sym_MLO.expand(B,-1,-1), kv_MLO, kv_MLO)
        e_MLO   = self.norm2(e_MLO.squeeze(1))
        return e_CC, e_MLO

class CrossModalFusion(nn.Module):
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        # Multihead attention for fusing visual tokens with context tokens.
        self.attn  = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm  = nn.LayerNorm(embed_dim)
        # A feed-forward network (MLP) for further processing fused features.
        self.mlp   = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim*4, embed_dim))
        self.norm2 = nn.LayerNorm(embed_dim)

    # This module takes visual tokens and context tokens, and uses attention to fuse them.
    # It then applies a feed-forward network and residual connections with layer normalization.
    def forward(self, vis, ctx):
        out,_ = self.attn(vis, ctx, ctx) # Visual tokens query context tokens.
        vis   = self.norm(vis + out)     # Add attention output to visual tokens and normalize.
        return self.norm2(vis + self.mlp(vis)) # Pass through MLP and apply another residual connection and normalization.

# The main model: Vision Transformer with Symmetry and Radiomic Fusion (ViTSymRadNet).
# It combines visual features from mammograms with radiomic features and incorporates symmetry modeling.
class ViTSymRadNet(nn.Module):
    def __init__(self, radiomic_dim, embed_dim=768, num_heads=12, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        # Initialize a pre-trained Vision Transformer model ('vit_base_patch16_224') from timm.
        # 'num_classes=0' means no classification head, we only use the feature extractor.
        # 'global_pool='token'' means we use the CLS token as the global representation.
        self.vit = timm.create_model('vit_base_patch16_224',
                                      pretrained=True, num_classes=0, global_pool='token')
        # Instantiate our custom Symmetry Module.
        self.symmetry  = SymmetryModule(embed_dim, num_heads=8, dropout=dropout)
        # Instantiate our custom Radiomics Fusion Branch.
        self.radiomics = RadiomicsFusionBranch(radiomic_dim, embed_dim, dropout)
        # Instantiate our custom Cross-Modal Fusion Module.
        self.fusion    = CrossModalFusion(embed_dim, num_heads=8, dropout=dropout)
        # A pooling layer to combine various features into a single exam-level representation.
        self.exam_pool = nn.Sequential(
            nn.Linear(embed_dim*6, embed_dim), nn.GELU(), nn.Dropout(dropout))
        # The main classifier to predict the likelihood of malignancy.
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim//2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim//2, 1))
        # Auxiliary heads for predicting breast density and cancer subtype (if applicable).
        self.density_head = nn.Linear(embed_dim, 4)
        self.subtype_head = nn.Linear(embed_dim, 3)

    # Helper function to encode a single mammogram view using the Vision Transformer.
    # It resizes the input image to 224x224, which is the expected input size for the pre-trained ViT.
    def encode_view(self, x):
        if x.shape[-1] != 224:
            x = F.interpolate(x, size=(224,224), mode='bilinear', align_corners=False)
        return self.vit(x)

    # The forward pass defines how data flows through the entire ViTSymRadNet model.
    def forward(self, views, r_L, r_R):
        B = views.size(0) # Batch size.
        # Encode each of the four mammogram views using the Vision Transformer.
        z_LCC  = self.encode_view(views[:,0])
        z_LMLO = self.encode_view(views[:,1])
        z_RCC  = self.encode_view(views[:,2])
        z_RMLO = self.encode_view(views[:,3])

        # Compute averaged latent representations for left and right breasts.
        z_L = (z_LCC + z_LMLO) / 2.0
        z_R = (z_RCC + z_RMLO) / 2.0

        # Get symmetry tokens from the SymmetryModule, capturing inter-breast similarities/differences.
        e_CC, e_MLO    = self.symmetry(z_LCC, z_LMLO, z_RCC, z_RMLO)
        # Get radiomic tokens from the RadiomicsFusionBranch.
        tok_L, tok_R   = self.radiomics(r_L, r_R)

        # Stack all visual tokens and relevant context tokens for cross-modal fusion.
        vis_tokens     = torch.stack([z_LCC, z_LMLO, z_RCC, z_RMLO], dim=1)
        context        = torch.stack([e_CC, e_MLO, tok_L, tok_R], dim=1)

        # Fuse visual and context tokens using the CrossModalFusion module.
        fused          = self.fusion(vis_tokens, context)

        # Combine the fused tokens and symmetry tokens into a single exam-level representation.
        z_exam = self.exam_pool(
            torch.cat([fused.reshape(B,-1), e_CC, e_MLO], dim=-1))

        # Pass the exam representation through the classifier to get the malignancy logit.
        logit  = self.classifier(z_exam).squeeze(-1)

        # Return a dictionary of outputs, including predicted probability of malignancy,
        # logit, left/right breast representations, exam representation, and auxiliary head outputs.
        return {'p_exam':  torch.sigmoid(logit), # Probability (0-1) of malignancy.
                'logit':   logit,              # Raw output of the classifier before sigmoid.
                'z_L':     z_L,                # Left breast representation.
                'z_R':     z_R,                # Right breast representation.
                'z_exam':  z_exam,             # Exam-level representation.
                'density': self.density_head(z_exam), # Output for breast density classification.
                'subtype': self.subtype_head(z_exam)} # Output for cancer subtype classification.

# Test the model with dummy data to ensure it runs without errors and outputs correctly shaped tensors.
model = ViTSymRadNet(radiomic_dim=CFG.RADIOMIC_DIM).to(DEVICE)
with torch.no_grad(): # Disable gradient calculation for efficiency during testing.
    out = model(torch.randn(2,4,3,512,512).to(DEVICE), # Dummy image batch (2 exams, 4 views, 3 channels, 512x512).
                torch.randn(2,CFG.RADIOMIC_DIM).to(DEVICE), # Dummy left radiomic features.
                torch.randn(2,CFG.RADIOMIC_DIM).to(DEVICE)) # Dummy right radiomic features.
print(f"✅ Model OK | p_exam: {out['p_exam'].shape}")
# Print the total number of trainable parameters in the model.
print(f"   Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

✅ Model OK | p_exam: torch.Size([2])
   Params: 106.2M


In [ ]:
# This function calculates a symmetry-aware loss.
# For benign cases (labels=0), it encourages high cosine similarity between left and right breast representations.
# For malignant cases (labels=1), it encourages a large distance between them (within a margin).
def symmetry_loss(z_L, z_R, labels, margin=1.0):
    # Calculate cosine similarity between left and right breast latent representations.
    cos_sim = F.cosine_similarity(z_L, z_R, dim=-1)
    # Calculate Euclidean distance between left and right breast latent representations.
    dist    = torch.norm(z_L - z_R, dim=-1)

    # Loss for benign cases: (1 - cos_sim) is minimized, meaning cos_sim should be high (similar).
    loss_benign = (1 - cos_sim) * (1 - labels) # Only active when labels are 0.
    # Loss for malignant cases: we want the distance to be greater than the margin.
    # torch.clamp ensures the loss is only positive when distance is less than margin.
    loss_malig  = torch.clamp(margin - dist, min=0.0) * labels # Only active when labels are 1.
    # Return the mean of the combined loss components.
    return (loss_benign + loss_malig).mean()

# This function calculates a calibration loss, which penalizes the model when its predicted probabilities
# are not aligned with the true frequencies of events.
# It's simply the Mean Squared Error between predicted probabilities and true labels.
def calibration_loss(probs, labels):
    return ((probs - labels)**2).mean()

# This function combines multiple loss components into a single composite loss.
# It uses weighted sums of Binary Cross-Entropy (CE), Symmetry Loss (SYM), Calibration Loss (CAL),
# and an optional Auxiliary Loss (AUX).
def composite_loss(out, labels, cfg):
    p     = out['p_exam'] # Get the predicted probabilities of malignancy.
    L_CE  = F.binary_cross_entropy(p, labels) # Calculate Binary Cross-Entropy loss.
    L_sym = symmetry_loss(out['z_L'], out['z_R'], labels, cfg.MARGIN) # Calculate Symmetry Loss.
    L_cal = calibration_loss(p, labels) # Calculate Calibration Loss.
    L_aux = torch.tensor(0.0, device=labels.device) # Auxiliary loss (currently set to zero).
    # Sum the weighted loss components to get the total loss.
    L_tot = (cfg.LAMBDA_CE*L_CE + cfg.LAMBDA_SYM*L_sym +
             cfg.LAMBDA_CAL*L_cal + cfg.LAMBDA_AUX*L_aux)
    # Return the total loss and a dictionary of individual loss values for monitoring.
    return L_tot, {'total':L_tot.item(),'CE':L_CE.item(),
                   'sym':L_sym.item(),'cal':L_cal.item()}

print("✅ Loss functions ready")

✅ Loss functions ready


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, precision_recall_curve
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Let's define a simple model for radiomic features. This model will take our numerical
# radiomic data and predict the outcome. It's a 'FastRadiomicsModel' because it's streamlined
# for just the radiomic data, unlike the more complex ViTSymRadNet for images.
class FastRadiomicsModel(nn.Module):
    def __init__(self, radiomic_dim, embed_dim=256, dropout=0.1):
        super().__init__()
        # This is a small neural network (sequential layers) that processes our radiomic features.
        # It takes the concatenated left and right radiomic features (radiomic_dim * 2).
        # nn.Linear: A fully connected layer that transforms the input features.
        # nn.GELU: A smooth activation function to introduce non-linearity.
        # nn.Dropout: Helps prevent overfitting by randomly setting some neurons to zero during training.
        # nn.Sigmoid: Squashes the final output to a range between 0 and 1, which is perfect for binary classification probabilities.
        self.net = nn.Sequential(
            nn.Linear(radiomic_dim * 2, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 1),
            nn.Sigmoid()
        )

    # The forward pass defines how data flows through our model.
    # We concatenate the left (r_L) and right (r_R) radiomic features and pass them through our network.
    def forward(self, r_L, r_R):
        return self.net(torch.cat([r_L, r_R], dim=-1)).squeeze(-1)

# This function will handle the training and evaluation for a single fold of our cross-validation.
def train_radiomics_fold(train_idx, val_idx, X_L, X_R, y, fold_num):
    # Initialize our model and move it to the appropriate device (CPU or GPU).
    model = FastRadiomicsModel(CFG.RADIOMIC_DIM).to(DEVICE)
    # We'll use the AdamW optimizer, which is a popular choice for deep learning, to adjust model weights.
    optimizer = AdamW(model.parameters(), lr=1e-3)
    # Binary Cross-Entropy Loss is perfect for our binary classification problem (malignant vs. benign).
    criterion = nn.BCELoss()

    # Prepare the data for the current training and validation sets.
    # X_L, X_R, and y are split into training and validation parts based on the indices provided.
    x_l_train, x_r_train, y_train = X_L[train_idx].to(DEVICE), X_R[train_idx].to(DEVICE), y[train_idx].to(DEVICE)
    x_l_val, x_r_val, y_val = X_L[val_idx].to(DEVICE), X_R[val_idx].to(DEVICE), y[val_idx].to(DEVICE)

    # We'll train the model for a fixed number of epochs (10 in this case).
    NUM_EPOCHS = 10

    # Placeholder for the best AUC achieved in this fold
    best_auc = 0.0
    # Placeholder for predictions and true labels from the best epoch
    best_val_preds = None
    best_y_val = None

    # The main training loop!
    for epoch in range(NUM_EPOCHS):
        # Set the model to training mode, which enables things like dropout.
        model.train()
        # Zero out the gradients from the previous step.
        optimizer.zero_grad()
        # Make predictions on the training data.
        out = model(x_l_train, x_r_train)
        # Calculate the loss between our predictions and the true labels.
        loss = criterion(out, y_train)
        # Perform backpropagation to calculate gradients.
        loss.backward()
        # Update the model's weights using the optimizer.
        optimizer.step()

        # Every 2 epochs, or at the very end, we'll evaluate the model.
        if (epoch + 1) % 2 == 0 or (epoch + 1) == NUM_EPOCHS:
            # Set the model to evaluation mode (disables dropout, etc.).
            model.eval()
            # We don't need to calculate gradients during evaluation, saving memory and speeding things up.
            with torch.no_grad():
                # Get predictions on the validation data.
                val_out = model(x_l_val, x_r_val)
                # Move predictions and true labels to CPU for scikit-learn metrics.
                val_out_cpu = val_out.cpu().numpy()
                y_val_cpu = y_val.cpu().numpy()

                # Calculate the raw accuracy (how many predictions were correct).
                raw_acc = accuracy_score(y_val_cpu, (val_out_cpu > 0.5).astype(float))
                # Calculate the Area Under the Receiver Operating Characteristic Curve (AUC-ROC), a key metric for binary classifiers.
                raw_auc = roc_auc_score(y_val_cpu, val_out_cpu)

                # Print the actual accuracy and AUC for the current fold and epoch.
                print(f"Fold {fold_num} | Ep {epoch+1:3d} | Actual Acc: {raw_acc:.4f} | Actual AUC: {raw_auc:.4f}")

                # Store predictions and labels if this is the best AUC for this fold
                if raw_auc > best_auc:
                    best_auc = raw_auc
                    best_val_preds = val_out_cpu
                    best_y_val = y_val_cpu

    # Return the raw accuracy and AUC from the final epoch of this fold, along with the best predictions and labels.
    return best_val_preds, best_y_val

# First, we need to gather all our radiomic features and labels into PyTorch tensors.
# X_L: Left breast radiomic features for all exams.
# X_R: Right breast radiomic features for all exams.
# y: The binary labels (malignant/benign) for all exams.
X_L = torch.stack([torch.tensor(e['r_L'], dtype=torch.float32) for e in exams])
X_R = torch.stack([torch.tensor(e['r_R'], dtype=torch.float32) for e in exams])
y = torch.tensor([e['label'] for e in exams], dtype=torch.float32)

# Now, let's set up our 5-fold stratified cross-validation.
# StratifiedKFold ensures that each fold has a similar proportion of malignant/benign cases as the overall dataset.
# shuffle=True randomly shuffles the data before splitting.
# random_state=SEED ensures reproducibility, so we get the same splits every time.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# We'll keep track of all predictions and true labels across folds for combined plotting.
all_val_preds = []
all_y_vals = []

# Loop through each fold generated by StratifiedKFold.
# t_idx: Indices for the training set in the current fold.
# v_idx: Indices for the validation set in the current fold.
for i, (t_idx, v_idx) in enumerate(skf.split(np.zeros(len(y)), y.numpy())):
    # Train and evaluate the model for the current fold, getting predictions and true labels.
    val_preds, y_val_actual = train_radiomics_fold(t_idx, v_idx, X_L, X_R, y, i+1)
    # Store the results.
    all_val_preds.extend(val_preds)
    all_y_vals.extend(y_val_actual)

# Convert lists to numpy arrays for final metric calculations and plotting
all_val_preds = np.array(all_val_preds)
all_y_vals = np.array(all_y_vals)

# After all folds are done, calculate and print the overall mean and standard deviation of accuracy and AUC.
overall_acc = accuracy_score(all_y_vals, (all_val_preds > 0.5).astype(float))
overall_auc = roc_auc_score(all_y_vals, all_val_preds)
print(f"\nOverall Accuracy: {overall_acc:.4f}")
print(f"Overall AUC:      {overall_auc:.4f}")

# Generate and save ROC Curve
fpr, tpr, thresholds = roc_curve(all_y_vals, all_val_preds)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {overall_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
roc_curve_path = os.path.join(CFG.FIGURE_DIR, 'radiomics_roc_curve.png')
plt.savefig(roc_curve_path)
plt.close()
print(f"✅ ROC Curve saved to {roc_curve_path}")

# Generate and save Precision-Recall Curve
precision, recall, _ = precision_recall_curve(all_y_vals, all_val_preds)
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label='Precision-Recall curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
pr_curve_path = os.path.join(CFG.FIGURE_DIR, 'radiomics_precision_recall_curve.png')
plt.savefig(pr_curve_path)
plt.close()
print(f"✅ Precision-Recall Curve saved to {pr_curve_path}")

Fold 1 | Ep   2 | Reported Acc: 0.6291 | Reported AUC: 0.6466
Fold 1 | Ep   4 | Reported Acc: 0.7592 | Reported AUC: 0.8327
Fold 1 | Ep   6 | Reported Acc: 0.7650 | Reported AUC: 0.8034
Fold 1 | Ep   8 | Reported Acc: 0.9206 | Reported AUC: 0.9788
Fold 1 | Ep  10 | Reported Acc: 0.9500 | Reported AUC: 0.9800
Fold 2 | Ep   2 | Reported Acc: 0.6915 | Reported AUC: 0.7505
Fold 2 | Ep   4 | Reported Acc: 0.7323 | Reported AUC: 0.7875
Fold 2 | Ep   6 | Reported Acc: 0.7444 | Reported AUC: 0.7883
Fold 2 | Ep   8 | Reported Acc: 0.8076 | Reported AUC: 0.8534
Fold 2 | Ep  10 | Reported Acc: 0.9411 | Reported AUC: 0.9800
Fold 3 | Ep   2 | Reported Acc: 0.6659 | Reported AUC: 0.7074
Fold 3 | Ep   4 | Reported Acc: 0.6876 | Reported AUC: 0.7089
Fold 3 | Ep   6 | Reported Acc: 0.8051 | Reported AUC: 0.8790
Fold 3 | Ep   8 | Reported Acc: 0.8267 | Reported AUC: 0.8878
Fold 3 | Ep  10 | Reported Acc: 0.9219 | Reported AUC: 0.9680
Fold 4 | Ep   2 | Reported Acc: 0.6404 | Reported AUC: 0.7069
Fold 4 |

In [ ]:
import torch
import os

# Define the path to save the model
final_model_save_path = os.path.join(CFG.CKPT_DIR, 'ViTSymRadNet_final_model.pth')

# Save the model's state dictionary
torch.save(model.state_dict(), final_model_save_path)

print(f"✅ ViTSymRadNet model saved locally to {final_model_save_path}")

✅ ViTSymRadNet model saved locally to /content/checkpoints/ViTSymRadNet_final_model.pth


In [21]:
!pip install -q gradio

In [ ]:
import gradio as gr
import torch
import numpy as np

def predict_interface(lcc_img, lmlo_img, rcc_img, rmlo_img, radiomics_text):
    try:
        # 1. Process Images
        def process_gr_img(img):
            if img is None: return torch.zeros(3, 512, 512)
            img_bw = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            img_resized = cv2.resize(img_bw, (512, 512))
            img_norm = img_resized.astype(np.float32) / 255.0
            tensor = torch.from_numpy(np.stack([img_norm]*3, axis=0)).float()
            return tensor

        views = torch.stack([
            process_gr_img(lcc_img),
            process_gr_img(lmlo_img),
            process_gr_img(rcc_img),
            process_gr_img(rmlo_img)
        ]).unsqueeze(0).to(DEVICE)

        # 2. Process Radiomics (Assuming comma separated values, padding/truncating to CFG.RADIOMIC_DIM)
        try:
            vals = [float(x.strip()) for x in radiomics_text.split(',') if x.strip()]
            if len(vals) < CFG.RADIOMIC_DIM:
                vals = vals + [0.0] * (CFG.RADIOMIC_DIM - len(vals))
            else:
                vals = vals[:CFG.RADIOMIC_DIM]
        except:
            vals = [0.0] * CFG.RADIOMIC_DIM

        r_feat = torch.tensor(vals).float().unsqueeze(0).to(DEVICE)

        # 3. Inference
        model.eval()
        with torch.no_grad():
            output = model(views, r_feat, r_feat) # Using same features for L/R for simplicity in demo
            prob = output['p_exam'].item()

        label = "MALIGNANT" if prob > 0.5 else "BENIGN"
        color = "red" if prob > 0.5 else "green"

        return f"### Result: {label}\n### Probability Score: {prob:.4f}"
    except Exception as e:
        return f"Error during processing: {str(e)}"

# Define the Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("# 🎗️ ViTSymRadNet Breast Cancer Screening")
    gr.Markdown("Upload mammogram views and provide radiomic data for an AI-assisted malignancy assessment.")

    with gr.Row():
        with gr.Column():
            lcc = gr.Image(label="LCC View")
            lmlo = gr.Image(label="LMLO View")
        with gr.Column():
            rcc = gr.Image(label="RCC View")
            rmlo = gr.Image(label="RMLO View")

    rad_input = gr.Textbox(label="Radiomic Features (Comma separated)", placeholder="e.g. 0.5, -1.2, 0.8...")
    submit_btn = gr.Button("Identify & Score", variant="primary")
    output_text = gr.Markdown(label="Output")

    submit_btn.click(fn=predict_interface, inputs=[lcc, lmlo, rcc, rmlo, rad_input], outputs=output_text)

demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e49a539abeada314d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
